# ARC-AGI-2 Phase 1A baseline

This notebook is the complete Phase 1A submission runtime. It uses only the Python standard library and predicts each test input by identity. The default mode is `submission`; evaluation mode is selected explicitly with `ARC_MODE=evaluation`.

Submission mode reads only `arc-agi_test_challenges.json`, discovers task IDs and test counts from that file, validates two attempts per output, and writes `/kaggle/working/submission.json` on Kaggle. Local runs use the repository-root `artifacts/submission.json` unless `ARC_OUTPUT_PATH` is set.

The official evaluation is output-weighted: the highest score among the two attempts is taken for each test output, then divided by the total number of test outputs. See the [Kaggle evaluation overview](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2/overview/evaluation).

In [ ]:
from __future__ import annotations

import copy
import json
import os
import re
import time
from pathlib import Path
from typing import Any

MAX_GRID_SIDE = 30
TASK_ID_PATTERN = re.compile(r"[0-9a-f]{8}")


class ValidationError(ValueError):
    pass


def _require_exact_keys(value, expected, label):
    if type(value) is not dict or set(value) != set(expected):
        raise ValidationError(f"{label} must have exactly these keys: {expected}")


def validate_task_id(task_id):
    if type(task_id) is not str or TASK_ID_PATTERN.fullmatch(task_id) is None:
        raise ValidationError(f"invalid task ID: {task_id!r}")


def validate_grid(grid, label="grid"):
    if type(grid) is not list or not grid or len(grid) > MAX_GRID_SIDE:
        raise ValidationError(f"{label} must be a nonempty list of rows")
    width = None
    for row_number, row in enumerate(grid):
        if type(row) is not list or not row or len(row) > MAX_GRID_SIDE:
            raise ValidationError(f"{label} row {row_number} is malformed")
        if width is None:
            width = len(row)
        elif len(row) != width:
            raise ValidationError(f"{label} is not rectangular")
        for color in row:
            if type(color) is not int or not 0 <= color <= 9:
                raise ValidationError(f"{label} contains an invalid color")
    return len(grid), width


def validate_challenges(challenges):
    if type(challenges) is not dict or not challenges:
        raise ValidationError("challenges must be a nonempty object")
    for task_id, task in challenges.items():
        validate_task_id(task_id)
        _require_exact_keys(task, ("train", "test"), f"task {task_id}")
        if type(task["train"]) is not list or not task["train"]:
            raise ValidationError(f"task {task_id} has malformed train data")
        for pair_number, pair in enumerate(task["train"]):
            _require_exact_keys(pair, ("input", "output"), f"train pair {pair_number}")
            validate_grid(pair["input"], f"task {task_id} train input")
            validate_grid(pair["output"], f"task {task_id} train output")
        if type(task["test"]) is not list or not task["test"]:
            raise ValidationError(f"task {task_id} has malformed test data")
        for test_number, test in enumerate(task["test"]):
            _require_exact_keys(test, ("input",), f"test input {test_number}")
            validate_grid(test["input"], f"task {task_id} test input")


def validate_solutions(challenges, solutions):
    if type(solutions) is not dict or set(solutions) != set(challenges):
        raise ValidationError("solutions must contain exactly the challenge task IDs")
    for task_id, task in challenges.items():
        outputs = solutions[task_id]
        if type(outputs) is not list or len(outputs) != len(task["test"]):
            raise ValidationError(f"solution count mismatch for task {task_id}")
        for output_number, output in enumerate(outputs):
            validate_grid(output, f"task {task_id} solution {output_number}")


def validate_submission(challenges: dict[str, Any], submission: dict[str, Any]) -> None:
    validate_challenges(challenges)
    if type(submission) is not dict or set(submission) != set(challenges):
        raise ValidationError("submission must contain exactly the challenge task IDs")
    for task_id, task in challenges.items():
        predictions: list[dict[str, Any]] = submission[task_id]
        if type(predictions) is not list or len(predictions) != len(task["test"]):
            raise ValidationError(f"prediction count mismatch for task {task_id}")
        for output_number, prediction in enumerate(predictions):
            _require_exact_keys(
                prediction, ("attempt_1", "attempt_2"), f"prediction {output_number}"
            )
            validate_grid(prediction["attempt_1"], f"task {task_id} attempt_1")
            validate_grid(prediction["attempt_2"], f"task {task_id} attempt_2")


def identity_submission(challenges):
    validate_challenges(challenges)
    return {
        task_id: [
            {
                "attempt_1": copy.deepcopy(test["input"]),
                "attempt_2": copy.deepcopy(test["input"]),
            }
            for test in task["test"]
        ]
        for task_id, task in challenges.items()
    }


def score_submission(challenges, solutions, submission):
    validate_submission(challenges, submission)
    validate_solutions(challenges, solutions)
    correct_outputs = 0
    solved_tasks = 0
    task_pass_fractions = []
    total_outputs = 0
    for task_id, task in challenges.items():
        task_correct = 0
        for prediction, expected in zip(
            submission[task_id], solutions[task_id], strict=True
        ):
            if (
                prediction["attempt_1"] == expected
                or prediction["attempt_2"] == expected
            ):
                task_correct += 1
        task_outputs = len(task["test"])
        correct_outputs += task_correct
        total_outputs += task_outputs
        solved_tasks += task_correct == task_outputs
        task_pass_fractions.append(task_correct / task_outputs)
    output_accuracy = correct_outputs / total_outputs
    task_solve_rate = solved_tasks / len(challenges)
    task_averaged_pass_at_2 = sum(task_pass_fractions) / len(task_pass_fractions)
    return {
        "correct_outputs": correct_outputs,
        "total_outputs": total_outputs,
        "output_accuracy": output_accuracy,
        "official_score": output_accuracy,
        "solved_tasks": solved_tasks,
        "total_tasks": len(challenges),
        "task_solve_rate": task_solve_rate,
        "task_averaged_pass_at_2": task_averaged_pass_at_2,
    }


def read_json(path):
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, sort_keys=True)
        handle.write(chr(10))


def resolve_repo_root():
    cwd = Path.cwd()
    if (cwd / "data").is_dir():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "data").is_dir():
        return cwd.parent
    return cwd


def resolve_data_dir():
    configured = os.environ.get("ARC_DATA_DIR")
    if configured:
        return Path(configured)
    candidates = [Path("/kaggle/input/arc-prize-2026-arc-agi-2")]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(
            path for path in sorted(kaggle_input.iterdir()) if path.is_dir()
        )
    candidates.append(resolve_repo_root() / "data")
    for candidate in candidates:
        if (candidate / "arc-agi_test_challenges.json").exists() or (
            candidate / "arc-agi_evaluation_challenges.json"
        ).exists():
            return candidate
    raise FileNotFoundError("could not locate ARC-AGI-2 challenge data")


def load_inputs(mode, data_dir):
    if mode == "evaluation":
        challenges = read_json(Path(data_dir) / "arc-agi_evaluation_challenges.json")
        validate_challenges(challenges)
        solutions = read_json(Path(data_dir) / "arc-agi_evaluation_solutions.json")
        validate_solutions(challenges, solutions)
        return challenges, solutions
    if mode != "submission":
        raise ValueError("ARC_MODE must be 'submission' or 'evaluation'")
    challenges = read_json(Path(data_dir) / "arc-agi_test_challenges.json")
    validate_challenges(challenges)
    return challenges, None


def resolve_output_path():
    configured = os.environ.get("ARC_OUTPUT_PATH")
    if configured:
        return Path(configured)
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.is_dir():
        return kaggle_working / "submission.json"
    return resolve_repo_root() / "artifacts" / "submission.json"

## Run mode

Kaggle uses submission mode by default. For local evaluation, set `ARC_MODE=evaluation`; for a local placeholder submission, set `ARC_MODE=submission`. `ARC_DATA_DIR` and `ARC_OUTPUT_PATH` are optional overrides for local execution.

In [ ]:
mode = os.environ.get("ARC_MODE", "submission").strip().lower()
data_dir = resolve_data_dir()
started = time.perf_counter()
challenges, solutions = load_inputs(mode, data_dir)
submission = identity_submission(challenges)
validate_submission(challenges, submission)
output_path = resolve_output_path()
write_json(output_path, submission)
round_trip = read_json(output_path)
validate_submission(challenges, round_trip)
elapsed = time.perf_counter() - started
total_outputs = sum(len(task["test"]) for task in challenges.values())
print(f"mode: {mode}")
print(f"data directory: {data_dir}")
print(f"tasks: {len(challenges)}; test outputs: {total_outputs}")
print(f"validated JSON readback: {output_path}")
if mode == "evaluation":
    metrics = score_submission(challenges, solutions, round_trip)
    print(
        f"output accuracy: {metrics['correct_outputs']}/{metrics['total_outputs']} = "
        f"{metrics['output_accuracy']:.6f}"
    )
    print(
        f"task solve rate: {metrics['solved_tasks']}/{metrics['total_tasks']} = "
        f"{metrics['task_solve_rate']:.6f}"
    )
    print(f"task-averaged pass@2: {metrics['task_averaged_pass_at_2']:.6f}")
    print(f"official score (output-weighted): {metrics['official_score']:.6f}")
else:
    print("solutions loaded: no")
print(f"elapsed seconds: {elapsed:.6f}")